In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# libraries
!pip install mne --quiet
import mne, os, numpy as np, matplotlib.pyplot as plt

In [ ]:
import os
import mne
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# --- Configuration ---
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025'

event_id = {'Wd2N': 1, 'Wd2E': 2}
tmin, tmax = -0.1, 0.8
baseline = (tmin, 0)
notch_freq = 60

# --- Containers ---
wd2n_averages = []
wd2e_averages = []

# --- Load montage & region assignment ---
montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
positions = montage.get_positions()['ch_pos']

def cart_to_sph(x, y, z):
    azimuth = np.arctan2(y, x)
    elevation = np.arctan2(z, np.sqrt(x**2 + y**2))
    radius = np.sqrt(x**2 + y**2 + z**2)
    return azimuth, elevation, radius

spherical_coords = {
    ch: cart_to_sph(*pos) for ch, pos in positions.items() if ch.startswith('E')
}

channels_by_region = defaultdict(list)
for ch, (azi, ele, _) in spherical_coords.items():
    deg_azi = np.degrees(azi)
    deg_ele = np.degrees(ele)

    if deg_ele > 50:
        channels_by_region['Frontal'].append(ch)
    elif 20 < deg_ele <= 50:
        if deg_azi < -60:
            channels_by_region['Left Frontal'].append(ch)
        elif deg_azi > 60:
            channels_by_region['Right Frontal'].append(ch)
        else:
            channels_by_region['Fronto-Central'].append(ch)
    elif -20 <= deg_ele <= 20:
        if abs(deg_azi) < 30:
            channels_by_region['Central'].append(ch)
        elif deg_azi < -30:
            channels_by_region['Left Temporal'].append(ch)
        else:
            channels_by_region['Right Temporal'].append(ch)
    elif -50 <= deg_ele < -20:
        channels_by_region['Parietal'].append(ch)
    else:
        channels_by_region['Occipital'].append(ch)

channels_by_region = dict(channels_by_region)

# --- Process Subjects ---
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

for participant in participants:
    participant_path = os.path.join(root_dir, participant)
    epochs_list = []

    for file in os.listdir(participant_path):
        if file.endswith('.set'):
            file_path = os.path.join(participant_path, file)
            print(f"📥 Loading {file_path}")

            raw = mne.io.read_raw_eeglab(file_path, preload=True)
            raw.filter(l_freq=0.1, h_freq=None)
            raw.notch_filter(freqs=notch_freq)
            raw.set_montage(montage)

            events, event_dict = mne.events_from_annotations(raw)
            present_events = set(event_dict.keys())
            relevant_event_id = {k: v for k, v in event_id.items() if k in present_events}
            if not relevant_event_id:
                print(f"⚠ No relevant events in {file}")
                continue

            epochs = mne.Epochs(raw, events, event_id=relevant_event_id,
                                tmin=tmin, tmax=tmax,
                                baseline=baseline, preload=True, detrend=1)
            epochs_list.append(epochs)

    if epochs_list:
        combined_epochs = mne.concatenate_epochs(epochs_list)
        print(f"✅ Combined epochs for {participant}")

        if 'Wd2N' in combined_epochs.event_id:
            wd2n_avg = combined_epochs['Wd2N'].average()
            wd2n_averages.append(wd2n_avg)

        if 'Wd2E' in combined_epochs.event_id:
            wd2e_avg = combined_epochs['Wd2E'].average()
            wd2e_averages.append(wd2e_avg)

# --- Grand Averages ---
wd2n_grand = mne.grand_average(wd2n_averages)
wd2e_grand = mne.grand_average(wd2e_averages)

# --- Compute Region-wise ERPs ---
region_erp_data = {}

for region, ch_list in channels_by_region.items():
    picks_n = [wd2n_grand.ch_names.index(ch) for ch in ch_list if ch in wd2n_grand.ch_names]
    picks_e = [wd2e_grand.ch_names.index(ch) for ch in ch_list if ch in wd2e_grand.ch_names]

    wd2n_avg_data = wd2n_grand.data[picks_n].mean(axis=0)
    wd2e_avg_data = wd2e_grand.data[picks_e].mean(axis=0)

    region_erp_data[region] = (wd2n_avg_data, wd2e_avg_data)

# --- Plot ERPs by Region ---
fig, axs = plt.subplots(3, 3, figsize=(15, 10))
axs = axs.flatten()

for i, (region, (n_data, e_data)) in enumerate(sorted(region_erp_data.items())):
    axs[i].plot(wd2n_grand.times, n_data, label='Wd2N')
    axs[i].plot(wd2e_grand.times, e_data, label='Wd2E')
    axs[i].axvline(0, color='black', linestyle='--', linewidth=0.5)
    axs[i].set_title(region)
    axs[i].legend()
    axs[i].set_xlabel('Time (s)')
    axs[i].set_ylabel('Amplitude (µV)')

plt.tight_layout()
plt.suptitle('ERP by Region: Wd2N vs Wd2E (128-Channel Regional Averaging)', fontsize=16, y=1.02)
plt.show()




In [ ]:
import os
import mne

# Base directory with participant folders
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025'

# Event codes of interest
event_id = {'Wd2N': 1, 'Wd2E': 2}
tmin, tmax = -0.1, 0.8
baseline = (tmin, 0)

# Get list of participant folders
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

for participant in participants:
    participant_path = os.path.join(root_dir, participant)

    # Skip if already processed
    epoched_file_path = os.path.join(participant_path, f"{participant}_Wd2_epochs-epo.fif")
    if os.path.exists(epoched_file_path):
        print(f"⏭️  Skipping {participant} — already processed.")
        continue

    epochs_list = []

    for file in os.listdir(participant_path):
        if file.endswith('.set'):
            file_path = os.path.join(participant_path, file)
            print(f"🔄 Processing: {file_path}")

            # Load data
            raw = mne.io.read_raw_eeglab(file_path, preload=True)
2
            # Filtering
            raw.filter(l_freq=0.1, h_freq=None)
            raw.notch_filter(freqs=60)

            # Get events
            events, event_dict = mne.events_from_annotations(raw)
            present_events = set(event_dict.keys())
            relevant_event_id = {k: v for k, v in event_id.items() if k in present_events}

            if not relevant_event_id:
                print(f"⚠️ No relevant events found in {file}")
                continue

            # Epoching
            epochs = mne.Epochs(raw, events, event_id=relevant_event_id,
                                tmin=tmin, tmax=tmax,
                                baseline=baseline, preload=True, detrend=1)

            epochs_list.append(epochs)

    # Combine and save
    if epochs_list:
        combined_epochs = mne.concatenate_epochs(epochs_list)
        combined_epochs.save(epoched_file_path, overwrite=True)
        print(f"✅ Saved combined epochs for {participant}")
    else:
        print(f"❌ No valid epochs for {participant}")


In [ ]:
import os
import mne
import matplotlib.pyplot as plt

# Set root directory
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025'

# Initialize containers
all_wd2n_epochs = []
all_wd2e_epochs = []

# Loop over participants
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

for participant in participants:
    epoched_file = os.path.join(root_dir, participant, f"{participant}_Wd2_epochs-epo.fif")

    if not os.path.exists(epoched_file):
        print(f"⚠️ No epochs found for {participant}")
        continue

    print(f"📥 Loading: {epoched_file}")
    epochs = mne.read_epochs(epoched_file, preload=True)

    if 'Wd2N' in epochs.event_id:
        all_wd2n_epochs.append(epochs['Wd2N'])

    if 'Wd2E' in epochs.event_id:
        all_wd2e_epochs.append(epochs['Wd2E'])

# Output directory
output_dir = os.path.join(root_dir, 'Group_Level_Results')
os.makedirs(output_dir, exist_ok=True)

# Grand Average: Wd2N
if all_wd2n_epochs:
    wd2n_all = mne.concatenate_epochs(all_wd2n_epochs)
    wd2n_avg = wd2n_all.average()

    fig_n = wd2n_avg.plot()
    plt.suptitle('Grand Average: Wd2N')
    wd2n_avg.save(os.path.join(output_dir, 'grand_average_Wd2N-ave.fif'), overwrite=True)
    print("✅ Saved grand average for Wd2N")

# Grand Average: Wd2E
if all_wd2e_epochs:
    wd2e_all = mne.concatenate_epochs(all_wd2e_epochs)
    wd2e_avg = wd2e_all.average()

    fig_e = wd2e_avg.plot()
    plt.suptitle('Grand Average: Wd2E')
    wd2e_avg.save(os.path.join(output_dir, 'grand_average_Wd2E-ave.fif'), overwrite=True)
    print("✅ Saved grand average for Wd2E")
